## Demo 4: PhotoService

A photo shop. Twenty-four JPEGs with a price, customers, and the orders they place.

The first three demos read a file and wrote all of it into a database. This one asks two questions
they never had to:

* **Can a database hold the file itself?** Not a path to it, not a description of it - the bytes.
* **How do you move only what is new?** There is a running application on the other side of this
  one, and it keeps writing while we transfer.

### What data do we have?

In [1]:
import os

data_path = r"..\data\photoservice"

photos = sorted(file for file in os.listdir(data_path) if file.endswith(".jpg"))
print(f"{len(photos)} photos")

for photo in photos[:5]:
    print(f"{photo:25} {os.path.getsize(os.path.join(data_path, photo)) / 1024 / 1024:6.2f} MB")

24 photos
A6407344-2048.jpg           1.28 MB
A6407354-2048.jpg           1.16 MB
A6407362-2048.jpg           1.28 MB
A6407374-2048.jpg           1.01 MB
A6407376-2048.jpg           1.66 MB


The `photo` table in PostgreSQL was created with the container and already knows the names and the
prices. Its `image` column is empty - that is our job.

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

from connect_pg_instance import connect_pg_instance
from invoke_pg_query import invoke_pg_query

pg_connection = connect_pg_instance(
    instance="127.0.0.1",
    database="photoservice",
    username="photoservice",
    password="Passw0rd!"
)

[VERBOSE] Creating connection to instance [127.0.0.1]
[VERBOSE] Using password authentication
[VERBOSE] Opening connection
[VERBOSE] Returning connection object


In [3]:
invoke_pg_query(
    connection=pg_connection,
    query="SELECT id, name, price, image FROM photo ORDER BY id"
).head()

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 24 rows


,id,name,price,image
0,1,A6407344-2048.jpg,12.95,None
1,2,A6407354-2048.jpg,11.95,None
2,3,A6407362-2048.jpg,10.95,None
3,4,A6407374-2048.jpg,9.95,None
4,5,A6407376-2048.jpg,9.95,None


### Binary data is just data

This is the shortest section of the whole port, and that is the point.

`Get-Content -AsByteStream -Raw` becomes `Path.read_bytes()`. psycopg binds a Python `bytes` to a
`bytea` without being told anything, so there is no converter, no `setinputsizes`, no special case -
the same `invoke_pg_query` that has been running `SELECT`s since demo 2 takes a 4 MB JPEG as a
parameter value.

In [4]:
for file in sorted(Path(data_path).glob("*.jpg")):
    invoke_pg_query(
        connection=pg_connection,
        query="UPDATE photo SET image = :image WHERE name = :name",
        parameter_values={
            "name": file.name,
            "image": file.read_bytes()
        },
        enable_exception=True
    )

print("done")

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-quer

Did it arrive? A row count would not tell us - the rows were already there. The question is whether
the bytes in the column are the bytes in the file.

In [5]:
stored = invoke_pg_query(
    connection=pg_connection,
    query="SELECT name, length(image) AS bytes_in_database FROM photo ORDER BY id",
    as_type="dict"
)

for row in stored[:5]:
    on_disk = os.path.getsize(os.path.join(data_path, row["name"]))
    print(f"{row['name']:25} {row['bytes_in_database']:>9} in the database, {on_disk:>9} on disk")

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 24 rows
A6407344-2048.jpg           1344891 in the database,   1344891 on disk
A6407354-2048.jpg           1213888 in the database,   1213888 on disk
A6407362-2048.jpg           1344052 in the database,   1344052 on disk
A6407374-2048.jpg           1055433 in the database,   1055433 on disk
A6407376-2048.jpg           1740461 in the database,   1740461 on disk


### From PostgreSQL to SQL Server

`bytea` on one side, `VARBINARY(MAX)` on the other, and in between the same pair of functions that
streamed StackExchange tables in demo 2: a data reader on the source, `write_sql_table` on the
target. Neither of them knows or cares that the column holds a JPEG.

In [6]:
from connect_sql_instance import connect_sql_instance
from get_pg_data_reader import get_pg_data_reader
from invoke_sql_query import invoke_sql_query
from write_sql_table import write_sql_table

sql_connection = connect_sql_instance(
    instance="127.0.0.1",
    database="PhotoService",
    username="PhotoService",
    password="Passw0rd!"
)

[VERBOSE] Creating connection to instance [127.0.0.1]
[VERBOSE] Using SQL authentication
[VERBOSE] Disabling connection pooling
[VERBOSE] Opening connection
[VERBOSE] Returning connection object


In [7]:
create_query = """
CREATE TABLE dbo.photo
( id     INT
, name   VARCHAR(50)
, price  NUMERIC(5, 2)
, image  VARBINARY(MAX)
, CONSTRAINT photo_pk
  PRIMARY KEY (id)
)
"""

invoke_sql_query(connection=sql_connection, query=create_query)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1


One number is worth changing here. `batch_size` defaults to 1000, which is right for the small rows
of demo 2 - but a batch of 1000 photos would be several gigabytes held in memory at once. Twenty-four
rows of a few megabytes want a small batch, not a large one.

In [8]:
data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT id, name, price, image FROM photo ORDER BY id"
)

write_sql_table(
    connection=sql_connection,
    table="dbo.photo",
    data_reader=data_reader,
    data_reader_row_count=24,
    batch_size=5
)

[VERBOSE] Getting data reader for [SELECT id, name, price, image FROM photo ORDER BY id]
[VERBOSE] Returning data reader with 4 columns
[VERBOSE] Importing data into [dbo].[photo]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 5/24 rows inserted (20.8%) - 18 rows/sec
[VERBOSE] 10/24 rows inserted (41.7%) - 12 rows/sec
[VERBOSE] 15/24 rows inserted (62.5%) - 15 rows/sec
[VERBOSE] 20/24 rows inserted (83.3%) - 16 rows/sec
[VERBOSE] 24/24 rows inserted (100.0%) - 16 rows/sec
[VERBOSE] Bulk insert complete


And now the only check that actually proves it: take a photo back out of SQL Server, write it to
disk, and open it. If a single byte were lost on the way, the file would not be a JPEG any more.

In [9]:
photos = invoke_sql_query(
    connection=sql_connection,
    query="SELECT id, name, price, image FROM dbo.photo ORDER BY id",
    as_type="dict"
)

first = photos[0]
print(f"{first['name']}, {first['price']} EUR, {len(first['image'])} bytes")

Path("test.jpg").write_bytes(first["image"])

# os.startfile("test.jpg")

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 24 rows
A6407344-2048.jpg, 12.95 EUR, 1344891 bytes


1344891

In [10]:
original = Path(data_path, first["name"]).read_bytes()

print("identical to the file on disk:", Path("test.jpg").read_bytes() == original)

Path("test.jpg").unlink()

identical to the file on disk: True


### A running application on the other side

Everything so far moved data that was sitting still. The rest of this demo does not.

The `photoservice` container runs `docker/photoservice-app.py`: it invents a customer every minute,
an order every second, and pays and ships those orders. It writes into the same PostgreSQL database
we just read from, and it does not stop while we work.

```
docker compose logs -f photoservice
```

That application is itself a port - the sibling repository runs the same shop in PowerShell. Two
things about it are worth knowing here. It calls the very `lib/` functions this notebook calls, out
of the same directory, mounted into the container. And where the sibling archives its logging events
as JSON files on MinIO, this one prints them, because MinIO is not part of this repository.

In [11]:
for table in ["customer", "order_header", "order_detail"]:
    count = invoke_pg_query(
        connection=pg_connection,
        query=f"SELECT COUNT(*) FROM {table}",
        as_type="single_value"
    )
    print(f"{table:15} {count} rows")

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
customer        74 rows
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
order_header    3656 rows
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
order_detail    51688 rows


In [12]:
invoke_pg_query(
    connection=pg_connection,
    query="SELECT * FROM order_header ORDER BY id DESC LIMIT 5"
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 5 rows


,id,customer_id,created_at,updated_at,payment_uuid,shipment_uuid
0,3658,45,2026-08-06 18:17:40.479,None,None,None
1,3657,39,2026-08-06 18:17:39.449,None,None,None
2,3656,10,2026-08-06 18:17:38.421,None,None,None
3,3655,45,2026-08-06 18:17:37.386,None,None,None
4,3654,57,2026-08-06 18:17:36.367,None,None,None


The three tables on the SQL Server side. `customer` gets one column the source does not have,
`transfered_at`, which is the transfer writing down when it ran.

In [13]:
for query in [
    "CREATE TABLE dbo.customer (id INT, firstname VARCHAR(50), surname VARCHAR(50), city VARCHAR(50), email VARCHAR(200), transfered_at DATETIME2, CONSTRAINT customer_pk PRIMARY KEY (id))",
    "CREATE TABLE dbo.order_header (id INT, customer_id INT, created_at DATETIME2, updated_at DATETIME2, payment_uuid UNIQUEIDENTIFIER, shipment_uuid UNIQUEIDENTIFIER, CONSTRAINT order_header_pk PRIMARY KEY (id))",
    "CREATE TABLE dbo.order_detail (order_id INT, photo_id INT, quantity INT, price NUMERIC(7, 2), CONSTRAINT order_detail_pk PRIMARY KEY (order_id, photo_id))"
]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1


#### The simplest answer: move all of it, every time

`truncate_table=True` empties the target first. It always works, it is always correct, and it stops
being an option the moment the table is big enough to matter.

`NOW() AS transfered_at` is computed by PostgreSQL and travels with the row - the target column is
filled by the source query, not by the writer.

In [14]:
data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT id, firstname, surname, city, email, NOW() AS transfered_at FROM customer"
)

write_sql_table(
    connection=sql_connection,
    table="dbo.customer",
    data_reader=data_reader,
    truncate_table=True
)

[VERBOSE] Getting data reader for [SELECT id, firstname, surname, city, email, NOW() AS transfered_at FROM customer]
[VERBOSE] Returning data reader with 6 columns
[VERBOSE] Importing data into [dbo].[customer]
[VERBOSE] Creating cursor
[VERBOSE] Truncating table
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 74 rows inserted - 1149 rows/sec
[VERBOSE] Bulk insert complete


#### Move only what is new

Ask the target what it already has, then ask the source for everything after that. The whole
technique is one `MAX(id)` and one `WHERE`.

Run this cell twice with a minute in between: the second run transfers the customer the application
invented while you were reading.

In [15]:
target_id = invoke_sql_query(
    connection=sql_connection,
    query="SELECT ISNULL(MAX(id), 0) FROM dbo.customer",
    as_type="single_value"
)

print("the target has everything up to id", target_id)

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT id, firstname, surname, city, email, NOW() AS transfered_at FROM customer WHERE id > :id",
    parameter_values={"id": target_id}
)

write_sql_table(
    connection=sql_connection,
    table="dbo.customer",
    data_reader=data_reader
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
the target has everything up to id 74
[VERBOSE] Getting data reader for [SELECT id, firstname, surname, city, email, NOW() AS transfered_at FROM customer WHERE id > :id]
[VERBOSE] Returning data reader with 6 columns
[VERBOSE] Importing data into [dbo].[customer]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] Bulk insert complete


### Two tables that belong together

An order is a row in `order_header` and its rows in `order_detail`. Transferring them is where
"only what is new" gets interesting, because the application is still writing between our two
statements.

The `time.sleep(5)` is there to make that visible. It is not a fix for anything - it is the bug.

In [16]:
import time

target_id = invoke_sql_query(
    connection=sql_connection,
    query="SELECT ISNULL(MAX(id), 0) FROM dbo.order_header",
    as_type="single_value"
)

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT * FROM order_header WHERE id > :id",
    parameter_values={"id": target_id}
)
write_sql_table(connection=sql_connection, table="dbo.order_header", data_reader=data_reader)

time.sleep(5)

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT * FROM order_detail WHERE order_id > :id",
    parameter_values={"id": target_id}
)
write_sql_table(connection=sql_connection, table="dbo.order_detail", data_reader=data_reader)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
[VERBOSE] Getting data reader for [SELECT * FROM order_header WHERE id > :id]
[VERBOSE] Returning data reader with 6 columns
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 1000 rows inserted - 12675 rows/sec
[VERBOSE] 2000 rows inserted - 12991 rows/sec
[VERBOSE] 3000 rows inserted - 12881 rows/sec
[VERBOSE] 3669 rows inserted - 12029 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Getting data reader for [SELECT * FROM order_detail WHERE order_id > :id]
[VERBOSE] Returning data reader with 4 columns
[VERBOSE] Importing data into [dbo].[order_detail]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 1000 rows inserted - 15082 rows/sec
[VERBOSE] 2000 rows inserted - 15646 rows/sec
[VERBOSE] 3000 rows inserted - 15724 rows/sec
[VERBOSE] 4000 rows inserted - 15428 rows/sec
[VERBOSE] 5000 rows 

Count them. The details belong to more orders than the headers do, because five seconds' worth of
new orders arrived in between and only their details were picked up.

In [17]:
invoke_sql_query(
    connection=sql_connection,
    query="""
    SELECT (SELECT COUNT(DISTINCT id)       FROM dbo.order_header) AS headers
         , (SELECT COUNT(DISTINCT order_id) FROM dbo.order_detail) AS orders_in_details
    """
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows


,headers,orders_in_details
0,3669,3674


#### Naming the upper end as well

The fix is not to be quicker. It is to decide, once, which rows this transfer is about - and then
ask both queries for exactly those.

In [18]:
for query in ["TRUNCATE TABLE dbo.order_header", "TRUNCATE TABLE dbo.order_detail"]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

target_id = invoke_sql_query(
    connection=sql_connection,
    query="SELECT ISNULL(MAX(id), 0) FROM dbo.order_header",
    as_type="single_value"
)
source_id = invoke_pg_query(
    connection=pg_connection,
    query="SELECT COALESCE(MAX(id), 0) FROM order_header",
    as_type="single_value"
)

print(f"transferring orders {target_id + 1} to {source_id}")

boundaries = {"target_id": target_id, "source_id": source_id}

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT * FROM order_header WHERE id > :target_id AND id <= :source_id",
    parameter_values=boundaries
)
write_sql_table(connection=sql_connection, table="dbo.order_header", data_reader=data_reader)

time.sleep(5)

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT * FROM order_detail WHERE order_id > :target_id AND order_id <= :source_id",
    parameter_values=boundaries
)
write_sql_table(connection=sql_connection, table="dbo.order_detail", data_reader=data_reader)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
transferring orders 1 to 3684
[VERBOSE] Getting data reader for [SELECT * FROM order_header WHERE id > :target_id AND id <= :source_id]
[VERBOSE] Returning data reader with 6 columns
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 1000 rows inserted - 14433 rows/sec
[VERBOSE] 2000 rows inserted - 14470 rows/sec
[VERBOSE] 3000 rows inserted - 13792 rows/sec
[VERBOSE] 3684 rows inserted - 10773 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Getting data reader for [SELECT * FROM order_detail WHERE order_id > :target_id AND order_id <= :source_id]
[VERBOSE] Retur

In [19]:
invoke_sql_query(
    connection=sql_connection,
    query="""
    SELECT (SELECT COUNT(DISTINCT id)       FROM dbo.order_header) AS headers
         , (SELECT COUNT(DISTINCT order_id) FROM dbo.order_detail) AS orders_in_details
    """
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows


,headers,orders_in_details
0,3684,3684


### One unit of work, and the biggest difference in this repository

The two writes above land one after the other. Between them the target is visibly wrong - headers
without details - and if the second one fails it stays wrong.

The sibling wraps them in a transaction: `$connection.BeginTransaction()` gives it an object, and
every command it hands that object to belongs to that transaction.

**Python has no such object.** In DB-API the transaction belongs to the *connection*: it is opened
for you by the first statement and it ends when you commit the connection. There is nothing to pass
to a function, so `-Transaction` cannot be ported as a parameter.

What the `lib/` functions needed instead was a way to be told **not to commit**. Every one of them
committed unconditionally - `write_sql_table` after each batch, `invoke_sql_query` even after a
`SELECT` - so a transaction spanning two calls was impossible before this demo. They now take
`commit=True`, and with `commit=False` a function neither commits nor rolls back: the calls make up
one unit of work and the caller ends it.

In [20]:
target_id = invoke_sql_query(
    connection=sql_connection,
    query="SELECT ISNULL(MAX(id), 0) FROM dbo.order_header",
    as_type="single_value"
)
source_id = invoke_pg_query(
    connection=pg_connection,
    query="SELECT COALESCE(MAX(id), 0) FROM order_header",
    as_type="single_value"
)
boundaries = {"target_id": target_id, "source_id": source_id}

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT * FROM order_header WHERE id > :target_id AND id <= :source_id",
    parameter_values=boundaries
)
write_sql_table(connection=sql_connection, table="dbo.order_header", data_reader=data_reader, commit=False)

time.sleep(5)

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT * FROM order_detail WHERE order_id > :target_id AND order_id <= :source_id",
    parameter_values=boundaries
)
write_sql_table(connection=sql_connection, table="dbo.order_detail", data_reader=data_reader, commit=False)

time.sleep(5)

sql_connection.commit()

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
[VERBOSE] Getting data reader for [SELECT * FROM order_header WHERE id > :target_id AND id <= :source_id]
[VERBOSE] Returning data reader with 6 columns
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 15 rows inserted - 2922 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Getting data reader for [SELECT * FROM order_detail WHERE order_id > :target_id AND order_id <= :source_id]
[VERBOSE] Returning data reader with 4 columns
[VERBOSE] Importing data into [dbo].[order_detail]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 206 rows inserted - 4246 rows/sec
[VERBOSE] Bulk insert complete


#### And the source has a transaction too

The upper bound above fixed the gap by naming it. A transaction on the *source* fixes it differently:
under `REPEATABLE READ` both queries see the same snapshot of the database, so the second one cannot
see an order the first one missed - and the upper bound is no longer needed.

This is the one place where Python is not just different but nicer to read. psycopg has a real
context manager: `with pg_connection.transaction():` commits at the end of the block and rolls back
if the block raises. That is the shape of the sibling's four lines of `BeginTransaction` / `Commit` /
`Dispose`, without any of them.

It also forces into the open something every cell above quietly got away with. A data reader is a
cursor, a cursor opens a transaction on its connection, and `write_sql_table` closing the cursor it
was handed does **not** end that transaction. Every transfer so far has left `pg_connection` sitting
*idle in transaction*, holding a read lock on the table it streamed. Nothing complained, because
nothing else wanted those tables.

The isolation level does complain. PostgreSQL will not change it while a transaction is running, and
without the `commit()` below, the cell fails with `can't change 'isolation_level' now: connection in
transaction status INTRANS`.

The sibling never meets this. An ADO.NET command without an explicit transaction commits itself, so
`Get-PgDataReader` leaves nothing behind.

In [21]:
import psycopg

for query in ["TRUNCATE TABLE dbo.order_header", "TRUNCATE TABLE dbo.order_detail"]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

target_id = invoke_sql_query(
    connection=sql_connection,
    query="SELECT ISNULL(MAX(id), 0) FROM dbo.order_header",
    as_type="single_value"
)

# Every data reader above opened a transaction on the source and left it open - closing
# the cursor is not the same as ending the transaction. Nothing has minded so far.
pg_connection.commit()

pg_connection.isolation_level = psycopg.IsolationLevel.REPEATABLE_READ

with pg_connection.transaction():
    data_reader = get_pg_data_reader(
        connection=pg_connection,
        query="SELECT * FROM order_header WHERE id > :target_id",
        parameter_values={"target_id": target_id}
    )
    write_sql_table(connection=sql_connection, table="dbo.order_header", data_reader=data_reader, commit=False)

    time.sleep(5)

    data_reader = get_pg_data_reader(
        connection=pg_connection,
        query="SELECT * FROM order_detail WHERE order_id > :target_id",
        parameter_values={"target_id": target_id}
    )
    write_sql_table(connection=sql_connection, table="dbo.order_detail", data_reader=data_reader, commit=False)

sql_connection.commit()

pg_connection.isolation_level = None

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
[VERBOSE] Getting data reader for [SELECT * FROM order_header WHERE id > :target_id]
[VERBOSE] Returning data reader with 6 columns
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 1000 rows inserted - 7307 rows/sec
[VERBOSE] 2000 rows inserted - 9527 rows/sec
[VERBOSE] 3000 rows inserted - 10690 rows/sec
[VERBOSE] 3711 rows inserted - 10890 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Getting data reader for [SELECT * FROM order_detail WHERE order_id > :target_id]
[VERBOSE] Returning data reader with 4 columns
[VERBOSE] Importing data into [dbo].[order_detail]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VER

In [22]:
invoke_sql_query(
    connection=sql_connection,
    query="""
    SELECT (SELECT COUNT(DISTINCT id)       FROM dbo.order_header) AS headers
         , (SELECT COUNT(DISTINCT order_id) FROM dbo.order_detail) AS orders_in_details
    """
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows


,headers,orders_in_details
0,3711,3711


### Rows that changed after they were transferred

New rows are the easy half. An order that was already transferred and has since been paid for is the
other half, and `MAX(id)` cannot find it - the id has not changed.

`updated_at` can. Ask the target for the newest one it has seen, and ask the source for everything
older than the last id but newer than that timestamp.

In [23]:
target_id = invoke_sql_query(
    connection=sql_connection,
    query="SELECT ISNULL(MAX(id), 0) FROM dbo.order_header",
    as_type="single_value"
)
target_last_updated = invoke_sql_query(
    connection=sql_connection,
    query="SELECT MAX(updated_at) FROM dbo.order_header",
    as_type="single_value"
)

print("orders up to id", target_id, "last update seen", target_last_updated)

updated_rows = invoke_pg_query(
    connection=pg_connection,
    query="SELECT * FROM order_header WHERE id <= :id AND updated_at > :updated_at",
    parameter_values={"id": target_id, "updated_at": target_last_updated}
)

print(len(updated_rows), "orders changed since then")
updated_rows.head()

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
orders up to id 3711 last update seen 2026-08-06 18:18:35.834000
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 26 rows
26 orders changed since then


,id,customer_id,created_at,updated_at,payment_uuid,shipment_uuid
0,2265,46,2026-08-06 17:53:30.568,2026-08-06 18:18:42.974,66b412b4-b7ad-4ec6-8d56-00c00dc1ded2,80255a0b-da10-4aa2-8ec2-18a632ada9f6
1,2673,49,2026-08-06 18:00:34.533,2026-08-06 18:18:48.081,0a815b96-5bc6-416c-8a71-312099c48099,324be5ae-af77-47c5-9e99-1030b0cc5754
2,2786,33,2026-08-06 18:02:31.681,2026-08-06 18:18:45.009,cd0b836f-012e-46f1-a022-baf2eb487a82,a0525cfe-9a96-4409-a605-09d430993d48
3,1856,2,2026-08-06 17:46:23.317,2026-08-06 18:18:37.877,44938675-19fe-422e-b4f9-ddbff825bbc2,1c18fcc7-0cc1-4458-b3ce-5e1b762dc262
4,3022,27,2026-08-06 18:06:37.373,2026-08-06 18:18:39.912,851dac54-5d60-4e4d-8e2e-58b582a35c04,f155bfd2-8fd9-47e2-bf9f-a429420bb60e


Two ways to get them into the target, and the choice is not about correctness.

Delete and rewrite is one statement per row plus one bulk write, and it works whatever changed.

In [24]:
for row in updated_rows.itertuples():
    invoke_sql_query(
        connection=sql_connection,
        query="DELETE dbo.order_header WHERE id = @id",
        parameter_values={"id": row.id},
        commit=False
    )

write_sql_table(
    connection=sql_connection,
    table="dbo.order_header",
    data=updated_rows,
    commit=False
)

sql_connection.commit()

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-quer

`UPDATE` names the columns that actually change, and never leaves the row missing - but it has to
know what those columns are.

In [25]:
update_query = """
UPDATE dbo.order_header
   SET updated_at    = @updated_at
     , payment_uuid  = @payment_uuid
     , shipment_uuid = @shipment_uuid
 WHERE id = @id
"""

for row in updated_rows.itertuples():
    invoke_sql_query(
        connection=sql_connection,
        query=update_query,
        parameter_values={
            "updated_at": row.updated_at,
            "payment_uuid": row.payment_uuid,
            "shipment_uuid": row.shipment_uuid,
            "id": row.id
        },
        commit=False
    )

sql_connection.commit()

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-quer

### Change Data Capture

Everything above works out for itself what changed, by comparing the two sides. SQL Server can be
asked to write it down instead: CDC records every insert, update and delete into a shadow table, and
a transfer reads that instead of guessing.

The database was created with `sys.sp_cdc_enable_db` already run - see
`docker/sqlserver-photoservice.sql` - so only the tables have to be enabled. This needs SQL Server
Agent, which is why `MSSQL_AGENT_ENABLED` is set in `docker-compose.yaml`.

In [26]:
for table in ["customer", "order_header", "order_detail"]:
    invoke_sql_query(
        connection=sql_connection,
        query=f"EXEC sys.sp_cdc_enable_table @source_schema = N'dbo', @source_name = N'{table}', @role_name = NULL",
        enable_exception=True
    )

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1


Two things decide whether there is anything to see. The capture job polls, so nothing appears the
instant a row is written - and CDC only records what happened *after* the table was enabled, which
was three lines ago.

So watch the orders rather than the customers. The application invents a customer once a minute and an
order once a second, and this cell does not run for a minute.

`__$operation` says what CDC saw: `2` is an insert, `1` a delete, and `3` and `4` are the before and
after image of an update.

In [27]:
target_id = invoke_sql_query(
    connection=sql_connection,
    query="SELECT ISNULL(MAX(id), 0) FROM dbo.order_header",
    as_type="single_value"
)

data_reader = get_pg_data_reader(
    connection=pg_connection,
    query="SELECT * FROM order_header WHERE id > :id",
    parameter_values={"id": target_id}
)
write_sql_table(connection=sql_connection, table="dbo.order_header", data_reader=data_reader)

time.sleep(10)

invoke_sql_query(
    connection=sql_connection,
    query="SELECT TOP 10 __$operation, id, customer_id, created_at FROM cdc.dbo_order_header_CT ORDER BY __$start_lsn DESC"
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows
[VERBOSE] Getting data reader for [SELECT * FROM order_header WHERE id > :id]
[VERBOSE] Returning data reader with 6 columns
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting rows from the data reader
[VERBOSE] 31 rows inserted - 2416 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 10 rows


,__$operation,id,customer_id,created_at
0,2,3742,35,2026-08-06 18:19:08.111
1,2,3741,50,2026-08-06 18:19:07.096
2,2,3740,1,2026-08-06 18:19:05.392
3,2,3739,2,2026-08-06 18:19:04.377
4,2,3738,45,2026-08-06 18:19:03.344
5,2,3737,44,2026-08-06 18:19:02.332
6,2,3736,23,2026-08-06 18:19:01.318
7,2,3735,27,2026-08-06 18:19:00.301
8,2,3734,41,2026-08-06 18:18:59.276
9,2,3733,9,2026-08-06 18:18:58.246


### Key takeaways

* Binary data is just data. `bytes` in Python, `bytea` in PostgreSQL, `VARBINARY(MAX)` in SQL Server,
  and not one line of conversion code between them - only a smaller `batch_size`, because the rows
  are megabytes rather than bytes.
* Transferring only what is new is one `MAX(id)` and one `WHERE`. Transferring it *correctly* while
  the source keeps writing is the hard part.
* Two queries against a moving source do not see the same database. Name the upper bound, or read
  both inside one transaction.
* An id finds new rows. Only a timestamp - or CDC - finds changed ones.
* A transaction in Python belongs to the connection, not to a command. That is why `lib/` grew
  `commit=False` rather than a `transaction` parameter.

### Cleanup

In [28]:
for table in ["customer", "order_header", "order_detail"]:
    invoke_sql_query(
        connection=sql_connection,
        query=f"EXEC sys.sp_cdc_disable_table @source_schema = N'dbo', @source_name = N'{table}', @capture_instance = N'all'",
        enable_exception=True
    )

for query in [
    "DROP TABLE dbo.photo",
    "DROP TABLE dbo.customer",
    "DROP TABLE dbo.order_header",
    "DROP TABLE dbo.order_detail"
]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

invoke_pg_query(connection=pg_connection, query="UPDATE photo SET image = NULL", enable_exception=True)

sql_connection.close()
pg_connection.close()

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=24
